# Configuration

Pydantic models for Isambard HPC settings. Defaults are loaded from
`config/isambard.toml`, with `ssh_host` from the `ISAMBARD_HOST`
environment variable (`.env` file).

In [ ]:
#|default_exp config

In [ ]:
#|export
import os
import tomllib
from dotenv import load_dotenv
from pydantic import BaseModel, model_validator

from ai_index.const import isambard_config_path

In [ ]:
#|export
def _load_config_toml() -> dict:
    """Load config/isambard.toml and return the [isambard] section."""
    with open(isambard_config_path, "rb") as f:
        data = tomllib.load(f)
    return data["isambard"]

In [ ]:
#|export
class IsambardConfig(BaseModel):
    """Configuration for Isambard HPC cluster access.

    Defaults are loaded from config/isambard.toml.
    """

    ssh_host: str
    ssh_user: str | None = None
    project_dir: str
    hf_cache_dir: str = "{project_dir}/hf_cache"
    logs_dir: str = "{project_dir}/logs"
    partition: str = "workq"
    default_gpus: int = 1
    default_cpus: int = 16
    default_mem: str = "80G"
    default_time: str = "12:00:00"
    cuda_module: str = "cudatoolkit/24.11_12.6"
    python_version: str = "3.12"
    torch_index_url: str = "https://download.pytorch.org/whl/cu126"

    @model_validator(mode="after")
    def _interpolate_paths(self):
        """Interpolate {project_dir} in path fields."""
        self.hf_cache_dir = self.hf_cache_dir.format(project_dir=self.project_dir)
        self.logs_dir = self.logs_dir.format(project_dir=self.project_dir)
        return self

    @classmethod
    def from_toml(cls, **overrides) -> "IsambardConfig":
        """Load config from bundled config.toml, merged with overrides.

        Requires ssh_host in the TOML or as an override.
        """
        defaults = _load_config_toml()
        defaults.update(overrides)
        return cls.model_validate(defaults)

    @classmethod
    def from_env(cls, **overrides) -> "IsambardConfig":
        """Load config from config.toml + env vars (.env file).

        Env vars: ISAMBARD_HOST (required), ISAMBARD_PROJECT_DIR (optional).
        """
        load_dotenv()
        ssh_host = os.environ.get("ISAMBARD_HOST")
        if not ssh_host:
            raise ValueError("ISAMBARD_HOST not set. Check your .env file.")
        env_overrides = {"ssh_host": ssh_host}
        project_dir = os.environ.get("ISAMBARD_PROJECT_DIR")
        if project_dir:
            env_overrides["project_dir"] = project_dir
        env_overrides.update(overrides)
        return cls.from_toml(**env_overrides)